In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
import numpy as np
from pixell import enmap, enplot, reproject
import glob
import matplotlib.pyplot as plt
import emcee, corner
from astropy.io import fits
import sys
from astropy import units as u, constants as const
sys.path.insert(0, '../src')
#sys.path.insert(0, "/home/gill/apps/szpack/python")
import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
import yaml
import itertools
from pixell import enmap

import bandpass as bp
import covariance as cov
import model
import utils as ut

import SZpack as SZ

from astropy.coordinates import SkyCoord
import astropy.units as u

from pixell import colorize
colorize.mpl_register("planck")

In [ ]:
# latex font sans-serif
plt.rcParams['font.family'] = 'sans-serif'
# font size 25
plt.rcParams['font.size'] = 20

config = "/home/gill/research/ACT/multi-freq-bridge/configs/case23_ajay.yaml"
cf = ut.get_config_file(config)
region = ut.get_region(cf['region_center_ra'], cf['region_center_dec'], cf['region_width'])
dire_data = "/home/gill/research/ACT/bridge/data_paper/data/data/act_no_reproj"

data_ref = ut.imap_dim_check(enmap.read_map(f"{dire_data}/act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits", 
                                            box=region))
apod_map = enmap.apod(data_ref*0+1, cf['apod_pix'])

apod_1D_line = apod_map[apod_map.shape[0]//2, :].copy()

# Create figure with two aligned subplots
fig, (ax2, ax1) = plt.subplots(2, 1, figsize=(6, 6), sharex=True, gridspec_kw={'height_ratios': [1, 10], 'hspace': 0})

# Top plot: apodization 1D line (smaller)
ax2.plot(apod_1D_line, 'k-', linewidth=2)
ax2.set_xlim(0, 250)
ax2.set_ylim(0, 1.1)
ax2.yaxis.tick_right()
ax2.yaxis.set_label_position('right')
ax2.text(0.5, 0.1, 'Apodization window', transform=ax2.transAxes, ha='center', va='bottom')

# Bottom plot: data reference (larger) - flipped vertically
im = ax1.imshow(np.fliplr(data_ref), cmap='planck', origin='lower', aspect='auto')
ax1.set_xlabel('Pixel (x)')
ax1.set_ylabel('Pixel (y)')

plt.tight_layout()
plt.savefig("../plots/apodization_window.pdf", dpi=500, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Your existing setup
config = "/home/gill/research/ACT/multi-freq-bridge/configs/case23_ajay.yaml"
cf = ut.get_config_file(config)
region = ut.get_region(cf['region_center_ra'], cf['region_center_dec'], cf['region_width'])
dire_data = "/home/gill/research/ACT/bridge/data_paper/data/data/act_no_reproj"

data_ref = ut.imap_dim_check(enmap.read_map(f"{dire_data}/act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits", 
                                            box=region))
apod_map = enmap.apod(data_ref*0+1, cf['apod_pix'])

# Method 1: Use subplots with shared x-axis to ensure same scale
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Plot the reference map
im = ax1.imshow(data_ref, origin='lower', cmap='viridis', aspect='auto')
ax1.set_title("Reference ACT Map")
ax1.set_ylabel("Pixel Y")

# Plot the apodization profile
mid_row_profile = apod_map[apod_map.shape[0]//2, :]
ax2.plot(np.arange(len(mid_row_profile)), mid_row_profile, 'r-', linewidth=2)
ax2.set_title("Line Across Apodization Map")
ax2.set_xlabel("Pixel X")
ax2.set_ylabel("Apodization Value")
ax2.set_ylim(0, 1.1)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Method 2: Overlay the profile location on the map
fig, ax = plt.subplots(figsize=(10, 8))

# Show the map
im = ax.imshow(apod_map, origin='lower', cmap='RdBu_r', vmin=0, vmax=1)
ax.set_title("Apodization Map with Profile Location")

# Add a horizontal line showing where the profile is taken
mid_row = apod_map.shape[0]//2
ax.axhline(y=mid_row, color='yellow', linestyle='--', linewidth=2, label=f'Profile at row {mid_row}')

# Add contours to show the apodization levels
contours = ax.contour(apod_map, levels=[0.1, 0.5, 0.9], colors='black', linewidths=1.5, alpha=0.7)
ax.clabel(contours, inline=True, fontsize=10)

ax.legend()
plt.colorbar(im, ax=ax, label='Apodization Value')
plt.show()

# Method 3: Side-by-side comparison with matching aspect
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: Full apodization map
im1 = ax1.imshow(apod_map, origin='lower', cmap='RdBu_r', vmin=0, vmax=1)
ax1.set_title("Apodization Map")
ax1.set_xlabel("Pixel X")
ax1.set_ylabel("Pixel Y")
mid_row = apod_map.shape[0]//2
ax1.axhline(y=mid_row, color='yellow', linestyle='--', linewidth=2)
plt.colorbar(im1, ax=ax1, label='Apodization Value')

# Right: Profile with aspect ratio indicator
mid_row_profile = apod_map[mid_row, :]
ax2.plot(mid_row_profile, 'r-', linewidth=2)
ax2.set_title(f"Profile at Row {mid_row}")
ax2.set_xlabel("Pixel X")
ax2.set_ylabel("Apodization Value")
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.1)

# Add vertical lines to show the actual width
apod_start = np.where(mid_row_profile > 0.01)[0][0]
apod_end = np.where(mid_row_profile > 0.01)[0][-1]
ax2.axvline(x=apod_start, color='green', linestyle=':', alpha=0.5, label='Apod edges')
ax2.axvline(x=apod_end, color='green', linestyle=':', alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# f(x) = 0.5*(1-np.cos(np.pi*x)) plot this from x = 0 to 2pi
x = np.linspace(0, 2*np.pi, 100)
y = 0.5 * (1 - np.cos(np.pi * x))
plt.figure(figsize=(8, 5))
plt.plot(x, y, label='Apodization Function')
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Apodization Function: $f(x) = 0.5(1 - \cos(\pi x))$")
plt.legend()
plt.grid()
plt.show()

In [ ]:
def map_maker_improved(mcmc_fname, cf_name, labels, labels_no_tex, burnin=1, thin=1, 
                       plot_converge=False, print_cf=False, plot_samples=False):
    
    from astropy import units as u
    from astropy.visualization.wcsaxes import SphericalCircle
    
    # Read chain samples and print diagnostics
    sampler = emcee.backends.HDFBackend(mcmc_fname)
    samples = sampler.get_chain(discard=burnin, flat=True, thin=thin)  
    samples_unflat = sampler.get_chain(discard=burnin)
    acc_frac = sampler.accepted / sampler.iteration
    
    print("*Average acceptance fraction is: {:.2f}%".format(np.mean(acc_frac)*100))
    
    ndim = len(labels)
    plt.rc('text', usetex=True)
    plt.rc('font', family='sans-serif', size=20)
    
    print("\nNumber of iterations: {:.0f}".format(samples.shape[0] / sampler.shape[0]))
   
    if plot_converge:
        print("Convergence plot")
        converge_plot(sampler, labels)
    
    cf = ut.get_config_file(cf_name)
    if print_cf:
        for key, value in cf.items():
            print(key, ":", value)    
    # Get region and common WCS from a reference ACT map
    region = ut.get_region(cf['region_center_ra'], cf['region_center_dec'], cf['region_width'])
    dire_data = "/home/gill/research/ACT/bridge/data_paper/data/data/act_no_reproj"
    dire_data_planck = "/home/gill/research/ACT/bridge/data_paper/data/data/planck_no_reproj"
   
    # Use one ACT map as the reference for Wcs and data_shape
    data_ref = ut.imap_dim_check(enmap.read_map(f"{dire_data}/act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits", 
                                               box=region))
    
    data_wcs = data_ref.wcs
    data_shape = data_ref.shape

    velocity_idx = [9, 19, 27]

    for idx, label in enumerate(labels):
        if (idx in velocity_idx):
            samples[:, idx] *= -1

    # Compute theta from samples (for all parameters, here for each label)
    theta = []
    for idx, label in enumerate(labels):
        data = samples[:, idx]

        mcmc_run = np.percentile(data, [16, 50, 84])
        err = 0.5 * (mcmc_run[2] - mcmc_run[0])
        n = len(data)
        
        iqr = np.percentile(data, 75) - np.percentile(data, 25)
       
        bin_width = 2 * iqr * n**(-1/3)
       
        num_bins = int((np.max(data) - np.min(data)) / bin_width)
        hist, bin_edges = np.histogram(data, bins=num_bins, density=True)
       
        mode_bin = np.argmax(hist)
        mode_value = 0.5*(bin_edges[mode_bin] + bin_edges[mode_bin+1])

        theta.append(mcmc_run[1])  # Use median as theta

        print("{}: {:.6f}, +/- {:.6f}".format(label, mcmc_run[1], err))
    
    # Prepare cluster and filament models
    print(theta)
    
    c1 = model.Cluster(theta=theta, name="abell401", model_choice="fit_vavg")
    c2 = model.Cluster(theta=theta, name="abell399", model_choice="fit_vavg")
    fil = model.Filament(theta=theta, model_choice="fit_vavg")

    ra_c1_idx = 0
    dec_c1_idx = 1
    ra_c2_idx = 10
    dec_c2_idx = 11
    ra_fil_idx = 20
    dec_fil_idx = 21

    ra_c1_fit = theta[ra_c1_idx]
    dec_c1_fit = theta[dec_c1_idx]
    ra_c2_fit = theta[ra_c2_idx]
    dec_c2_fit = theta[dec_c2_idx]
    ra_fil_fit = theta[ra_fil_idx]
    dec_fil_fit = theta[dec_fil_idx]

    # Set up frequency-dependent dictionary: keys as string frequencies
        # Set up frequency-dependent dictionary: keys as string frequencies
    freqs_params = {

        '30_npipe': {'freq':30, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
                    'file':"planck_npipe_030_coadd_map_srcfree.fits"},
        '44_npipe': {'freq':44, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
                    'file':"planck_npipe_044_coadd_map_srcfree.fits"},
        '70_npipe': {'freq':70, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
                    'file':"planck_npipe_070_coadd_map_srcfree.fits"},
        '100_npipe':{'freq':100, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
                    'file':"planck_npipe_100_coadd_map_srcfree.fits"},
        '143_npipe':{'freq':143, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
                    'file':"planck_npipe_143_coadd_map_srcfree.fits"},
        '217_npipe':{'freq':217, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
                    'file':"planck_npipe_217_coadd_map_srcfree.fits"},
        '353_npipe':{'freq':353, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
                    'file':"planck_npipe_353_coadd_map_srcfree.fits"},
        '545_npipe':{'freq':545, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
                    'file':"planck_npipe_545_coadd_map_srcfree.fits"},

        '98_pa5': {'freq':98,  'array':'pa5',  'inst':'act',     'dir':dire_data, 
                    'file':"act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits"},
        '98_pa6': {'freq':98,  'array':'pa6',  'inst':'act',     'dir':dire_data, 
                            'file':"act_cut_dr6v2_pa6_f098_4way_coadd_map_srcfree.fits"},

        '150_pa5':{'freq':150, 'array':'pa5',  'inst':'act',     'dir':dire_data, 
                    'file':"act_cut_dr6v2_pa5_f150_4way_coadd_map_srcfree.fits"},
        '150_pa4': {'freq':150, 'array':'pa4',  'inst':'act',     'dir':dire_data, 
                                'file':"act_cut_dr6v2_pa4_f150_4way_coadd_map_srcfree.fits"},
        '150_pa6': {'freq':150, 'array':'pa6',  'inst':'act',     'dir':dire_data, 
                                'file':"act_cut_dr6v2_pa6_f150_4way_coadd_map_srcfree.fits"},
        '220_pa4':{'freq':220, 'array':'pa4',  'inst':'act',     'dir':dire_data, 
                    'file':"act_cut_dr6v2_pa4_f220_4way_coadd_map_srcfree.fits"}
    }
        
    ref_data_dict = {}
    model_tot_dict = {}
    ref_data_dict_uK = {}
    model_tot_dict_uK = {}

    xgrid, ygrid = np.meshgrid(np.arange(data_shape[1]), np.arange(data_shape[0]))
    
    for key, params in freqs_params.items():
        freq = params['freq']
        array = params['array']
        inst = params['inst']
        map_file = f"{params['dir']}/{params['file']}"
        
        ref_map = enmap.read_map(map_file, box=region)
        ref_data_curr = ut.imap_dim_check(ref_map)

        flux_factor = ut.flux_factor(array, freq)
        ref_data_curr *= flux_factor
        
        # Build SZ models using the same grid
        c1_model = c1.szmodel(frequency=freq, array=array, z=cf['c1_z'], muo=cf['c1_muo'],
                               xgrid=xgrid, ygrid=ygrid, ellipticity_type="numerator")
        c2_model = c2.szmodel(frequency=freq, array=array, z=cf['c2_z'], muo=cf['c2_muo'],
                               xgrid=xgrid, ygrid=ygrid, ellipticity_type="numerator")
        fil_model = fil.szmodel(frequency=freq, array=array, z=cf['fil_z'], muo=cf['fil_muo'],
                                 xgrid=xgrid, ygrid=ygrid)
        
        # CHANGE HERE
        total_model = c1_model[0] + c2_model[0] + fil_model[0]
        #total_model = fil_model[0]

        beam = ut.get_2d_beam(data_shape=ref_data_curr.shape, freq=freq, array=array, 
                              inst=inst, version="dr6v2", data_wcs=data_wcs)
        
        model_tot = np.real(np.fft.ifft2(np.fft.fft2(total_model) * beam)) 

        # convert model to microK
        model_tot_microK = model_tot / flux_factor
        ref_data_curr_microK = ref_data_curr / flux_factor

        ref_data_dict[key] = ref_data_curr
        model_tot_dict[key] = model_tot
        model_tot_dict_uK[key] = model_tot_microK
        ref_data_dict_uK[key] = ref_data_curr_microK

    # make a dictionary with vmin and vmax for each frequency
    vmin_vmax_dict = {}

    vmin_vmax_dict['30'] = {'vmin': -10, 'vmax': 10}
    vmin_vmax_dict['44'] = {'vmin': -15, 'vmax': 15}
    vmin_vmax_dict['70'] = {'vmin': -50, 'vmax': 50}
    vmin_vmax_dict['98'] = {'vmin': -60, 'vmax': 60}
    vmin_vmax_dict['100'] = {'vmin': -100, 'vmax': 100}
    vmin_vmax_dict['143'] = {'vmin': -150, 'vmax': 150}
    vmin_vmax_dict['150'] = {'vmin': -150, 'vmax': 150}
    vmin_vmax_dict['217'] = {'vmin': -300, 'vmax': 300}
    vmin_vmax_dict['220'] = {'vmin': -300, 'vmax': 300}
    vmin_vmax_dict['353'] = {'vmin': -1000, 'vmax': 1000}
    vmin_vmax_dict['545'] = {'vmin': 0, 'vmax': 3000}

    vmin_vmax_model_dict = {
        '30': {'vmin': -2, 'vmax': 2},
        '44': {'vmin': -6, 'vmax': 6},
        '70': {'vmin': -25, 'vmax': 25},
        '98': {'vmin': -80, 'vmax': 80},
        '100': {'vmin': -35, 'vmax': 35},
        '143': {'vmin': -40, 'vmax': 40},
        '150': {'vmin': -40, 'vmax': 40},
        '217': {'vmin': -30, 'vmax': 30},
        '220': {'vmin': -30, 'vmax': 30},
        '353': {'vmin': 0, 'vmax': 500},
        '545': {'vmin': 0, 'vmax': 500}
    }

    beam_fwhm_dict = 
    {
        '30': 32.4, 
        '44': 27.1, 
        '70': 13.3,
        '98': 2.1, 
        '100': 9.7, 
        '143': 7.3,
        '150': 1.4, 
        '217': 5.0, 
        '220': 1.0,
        '353': 4.9, 
        '545': 4.60
    } # units of arcmin
        
    # PLOTTING: Define instruments and array mapping for plot titles
    inst_dict = {'98':"ACT", "150":"ACT", "220":"ACT", 
                 "30":"Planck", "44":"Planck", "70":"Planck",
                 "100":"Planck", "143":"Planck", 
                 "217":"Planck", "353": "Planck", "545":"Planck"}
    array_dict = {'98':'pa5', "150":'pa5', "220":'pa4',
                  "30":'npipe', "44":'npipe', "70":'npipe',
                  "100":'npipe', "143":'npipe', "217":'npipe',
                  "353":'npipe', "545":'npipe'}
    
    # Order for plotting (only those keys present in freqs_params)
    plot_keys = ['30','44','70','98','100','143','150','217','220','353','545']
    
    for key, params in freqs_params.items():
        # Skip any frequencies not processed
        if key not in ref_data_dict:
            print(f"Skipping {key} GHz as it is not processed.")
            continue
        
        # Get data for this frequency
        ref_data_curr = ref_data_dict[key]  # Convert to Jy/sr
        model_tot = model_tot_dict[key]     # Convert to Jy/sr
        residual = ref_data_curr - model_tot

        model_tot_microK = model_tot_dict_uK[key] # uK
        ref_data_curr_microK = ref_data_dict_uK[key] # uK
        residual_microK = ref_data_curr_microK - model_tot_microK   

        vmin = None
        vmax = None
        vmin_model = None    
        vmax_model = None    
        
        # Create figure with individual subplots for better control
        fig = plt.figure(figsize=(15, 5))
        
        # Data panel
        ax1 = fig.add_subplot(131, projection=data_ref.wcs)
        im1 = ax1.imshow(ref_data_curr / 1e3, origin='lower', cmap='planck', 
                         vmin=vmin, vmax=vmax, interpolation='none')
        ax1.coords[0].set_axislabel('Right Ascension')
        ax1.coords[1].set_axislabel('Declination')
        ax1.coords[0].set_major_formatter('d')
        ax1.coords[1].set_major_formatter('d')
        ax1.invert_xaxis()
        ax1.set_title(f'Data: {key} GHz', fontsize=20, pad=20)
        
        # Add beam circle to data panel
        beam_radius_arcmin = beam_fwhm_dict[key.split('_')[0]] / 2
        # Calculate offset based on beam size (larger beams get larger offsets)
        # Scale the offset with beam size, with a minimum offset for small beams
        ra0, dec0 = cf['region_center_ra'] - 0.7, cf['region_center_dec'] + 0.7

        sky_center = SkyCoord(ra0, dec0, unit='deg', frame='icrs')

        beam_circle = SphericalCircle(sky_center, beam_radius_arcmin * u.arcmin,
                                transform=ax1.get_transform('icrs'),
                               edgecolor='black', facecolor='none', linewidth=2)
        ax1.add_patch(beam_circle)

        # Model panel
        ax2 = fig.add_subplot(132, projection=data_ref.wcs)
        im2 = ax2.imshow(model_tot / 1e3, origin='lower', cmap='planck', 
                         vmin=vmin_model, vmax=vmax_model, interpolation='none')
        ax2.coords[0].set_axislabel('Right Ascension')
        ax2.coords[1].set_axislabel('Declination')
        ax2.coords[0].set_major_formatter('d')
        ax2.coords[1].set_major_formatter('d')
        ax2.invert_xaxis()
        ax2.set_title(f'Model: {key} GHz', fontsize=20, pad=20)
        
        # Add beam circle to model panel
        beam_circle = SphericalCircle(sky_center, beam_radius_arcmin * u.arcmin,
                        transform=ax2.get_transform('icrs'),
                       edgecolor='black', facecolor='none', linewidth=2)
        ax2.add_patch(beam_circle)
        
        # Residual panel
        ax3 = fig.add_subplot(133, projection=data_ref.wcs)
        im3 = ax3.imshow(residual / 1e3, origin='lower', cmap='planck', 
                 vmin=vmin, vmax=vmax, interpolation='none')
        ax3.coords[0].set_axislabel('Right Ascension')
        ax3.coords[1].set_axislabel('Declination')
        ax3.coords[0].set_major_formatter('d')
        ax3.coords[1].set_major_formatter('d')
        ax3.invert_xaxis()
        ax3.set_title(f'Residual: {key} GHz', fontsize=20, pad=20)
        
        # Add beam circle to residual panel
        beam_circle = SphericalCircle(sky_center, beam_radius_arcmin * u.arcmin,
                        transform=ax3.get_transform('icrs'),
                       edgecolor='black', facecolor='none', linewidth=2)
        ax3.add_patch(beam_circle)
        
        # Add colorbars
        plt.colorbar(im1, ax=ax1, orientation='horizontal', pad=0.25, 
                     label=r'$I$ [kJy/sr]', fraction=0.046)
        plt.colorbar(im2, ax=ax2, orientation='horizontal', pad=0.25, 
                     label=r'$I$ [kJy/sr]', fraction=0.046)
        plt.colorbar(im3, ax=ax3, orientation='horizontal', pad=0.25, 
                     label=r'$I$ [kJy/sr]', fraction=0.046)
    
        ref_data_curr = enmap.ndmap(ref_data_curr, data_ref.wcs)
        model_tot = enmap.ndmap(model_tot, data_ref.wcs)
        residual = enmap.ndmap(residual, data_ref.wcs)
    
        ref_data_curr_microK = enmap.ndmap(ref_data_curr_microK, data_ref.wcs)
        model_tot_microK = enmap.ndmap(model_tot_microK, data_ref.wcs)
        residual_microK = enmap.ndmap(residual_microK, data_ref.wcs)
    
        #plt.savefig(f"../plots/bridge_indv_{key}.pdf", bbox_inches='tight', dpi=300, format='pdf')
        plt.show()

        # # save Jy signals
        # print(f"Saving maps for {key} GHz")
        # dire_save = "/home/gill/research/ACT/paper/models/indiv/aug17/total_model_all_three_systems_small_map/"
        # enmap.write_map(f"{dire_save}/maps/bridge_indv_{key}_data_Jy.fits", ref_data_curr)
        # enmap.write_map(f"{dire_save}/models/bridge_indv_{key}_model_Jy.fits", model_tot)
        # enmap.write_map(f"{dire_save}/residuals/bridge_indv_{key}_residual_Jy.fits", residual)

        # # save uK signals
        # enmap.write_map(f"{dire_save}/maps/bridge_indv_{key}_data_uK.fits", ref_data_curr_microK)
        # enmap.write_map(f"{dire_save}/models/bridge_indv_{key}_model_uK.fits", model_tot_microK)
        # enmap.write_map(f"{dire_save}/residuals/bridge_indv_{key}_residual_uK.fits", residual_microK)
    

In [ ]:
labels = [r'$RA_{\rm A401}$', 
            r'$DEC_{\rm A401}$', 
            r'$\beta_{\rm A401}$', 
            r'$r_{\rm c, A401}$ [$^{\prime}$]', 
            r'$e_{\rm A401}$', 
            r'$\theta_{\rm A401}$', 
            r'$\tau_{\rm A401}$', 
            r'$T_{\rm e, A401}$', 
            r'$A_{\rm D, A401}$',
            r"$v_{r, A401}$",
            
            r'$RA_{\rm A399}$', 
            r'$DEC_{\rm A399}$', 
            r'$\beta_{\rm A399}$', 
            r'$r_{\rm c, A399}$ [$^{\prime}$]', 
            r'$e_{\rm A399}$', 
            r'$\theta_{\rm A399}$', 
            r'$\tau_{\rm A399}$', 
            r'$T_{\rm e, A399}$', 
            r'$A_{\rm D, A399}$',
            r"$v_{r, A399}$",

            r"$RA_{\rm fil}$",
            r"$DEC_{\rm fil}$",
            r"$L_{\rm fil}$",
            r"$W_{\rm fil}$",
            r'$\tau_{\rm fil}$',
            r'$T_{\rm e, fil}$',
            r'$A_{\rm D, fil}$',
            
            r"$v_{r, fil}$"]

labels_no_tex = [
    "RA_A401",
    "DEC_A401",
    "beta_A401",
    "r_c_A401 [']",
    "e_A401",
    "theta_A401",
    "tau_A401",
    "T_e_A401",
    "A_D_A401",
    
    "RA_A399",
    "DEC_A399",
    "beta_A399",
    "r_c_A399 [']",
    "e_A399",
    "theta_A399",
    "tau_A399",
    "T_e_A399",
    "A_D_A399",
    
    "RA_fil",
    "DEC_fil",
    "L_fil",
    "W_fil",
    "tau_fil",
    "T_e_fil",
    "A_D_fil",
    
    "v_r_avg"
]

config = "/home/gill/research/ACT/multi-freq-bridge/configs/case23_ajay.yaml"
chain = "/home/gill/research/ACT/bridge/results/ajay/final_runs/case23/chain.h5"


_ = map_maker_improved_cols(mcmc_fname=chain,
                       cf_name=config,
                       labels=labels,
                       labels_no_tex=labels_no_tex,
                       burnin=70000,
                       thin=100, 
                       plot_converge=False,
                       plot_samples=0)


In [ ]:
labels = [r'$RA_{\rm A401}$', 
            r'$DEC_{\rm A401}$', 
            r'$\beta_{\rm A401}$', 
            r'$r_{\rm c, A401}$ [$^{\prime}$]', 
            r'$e_{\rm A401}$', 
            r'$\theta_{\rm A401}$', 
            r'$\tau_{\rm A401}$', 
            r'$T_{\rm e, A401}$', 
            r'$A_{\rm D, A401}$',
            r"$v_{r, A401}$",
            
            r'$RA_{\rm A399}$', 
            r'$DEC_{\rm A399}$', 
            r'$\beta_{\rm A399}$', 
            r'$r_{\rm c, A399}$ [$^{\prime}$]', 
            r'$e_{\rm A399}$', 
            r'$\theta_{\rm A399}$', 
            r'$\tau_{\rm A399}$', 
            r'$T_{\rm e, A399}$', 
            r'$A_{\rm D, A399}$',
            r"$v_{r, A399}$",

            r"$RA_{\rm fil}$",
            r"$DEC_{\rm fil}$",
            r"$L_{\rm fil}$",
            r"$W_{\rm fil}$",
            r'$\tau_{\rm fil}$',
            r'$T_{\rm e, fil}$',
            r'$A_{\rm D, fil}$',
            
            r"$v_{r, fil}$"]

labels_no_tex = [
    "RA_A401",
    "DEC_A401",
    "beta_A401",
    "r_c_A401 [']",
    "e_A401",
    "theta_A401",
    "tau_A401",
    "T_e_A401",
    "A_D_A401",
    
    "RA_A399",
    "DEC_A399",
    "beta_A399",
    "r_c_A399 [']",
    "e_A399",
    "theta_A399",
    "tau_A399",
    "T_e_A399",
    "A_D_A399",
    
    "RA_fil",
    "DEC_fil",
    "L_fil",
    "W_fil",
    "tau_fil",
    "T_e_fil",
    "A_D_fil",
    
    "v_r_avg"
]

config = "/home/gill/research/ACT/multi-freq-bridge/configs/case23_ajay.yaml"
chain = "/home/gill/research/ACT/bridge/results/ajay/final_runs/case23/chain.h5"


_ = map_maker_improved_fullSize(mcmc_fname=chain,
                       cf_name=config,
                       labels=labels,
                       labels_no_tex=labels_no_tex,
                       burnin=50000,
                       thin=1, 
                       plot_converge=False,
                       plot_samples=0)
